# Phase 1: Data Cleaning

Loads the raw CSVs from the `datasets` folder, cleans them, merges them, and outputs a single `clean_matches.csv` for EDA.

In [1]:
import pandas as pd
import numpy as np
import os
import io

DATA_DIR = '../datasets/League of Legends Ranked Matches'
CHAMP_DIR = '../datasets/League of Legends Champion'
OUTPUT_DIR = '../data/processed'

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Load Data

In [2]:
# Load core match data
matches = pd.read_csv(f'{DATA_DIR}/matches.csv')
participants = pd.read_csv(f'{DATA_DIR}/participants.csv')
stats1 = pd.read_csv(f'{DATA_DIR}/stats1.csv')
stats2 = pd.read_csv(f'{DATA_DIR}/stats2.csv')
teamstats = pd.read_csv(f'{DATA_DIR}/teamstats.csv')

# Load champion mapping
champs_old = pd.read_csv(f'{DATA_DIR}/champs.csv')
champs_new = pd.read_csv(f'{CHAMP_DIR}/League of legend Champions 2024.csv')

print(f"Loaded {len(matches)} matches, {len(participants)} participants, and {len(champs_new)} new champions.")


C:\Users\asus\AppData\Local\Temp\ipykernel_13752\122443584.py:5: DtypeWarning: Columns (0: wardsbought) have mixed types. Specify dtype option on import or set low_memory=False.
  stats2 = pd.read_csv(f'{DATA_DIR}/stats2.csv')


Loaded 184069 matches, 1834520 participants, and 168 new champions.


## 2. Prepare Champion Data
The new champion dataset has names that might slightly differ from the old one, and it contains the metadata we need (Classes, Role, Difficulty).

In [3]:
# Clean champion names for better mapping
def clean_champ_name(name):
    if not isinstance(name, str): return name
    return name.lower().replace(" ", "").replace("'", "").replace(".", "")

champs_old['name_clean'] = champs_old['name'].apply(clean_champ_name)
champs_new['name_clean'] = champs_new['Name'].apply(clean_champ_name)

# Identify unmapped ones and fix (e.g. Wukong vs MonkeyKing)
champs_old.loc[champs_old['name'] == 'MonkeyKing', 'name_clean'] = 'wukong'
champs_old.loc[champs_old['name'] == 'ChoGath', 'name_clean'] = 'chogath'
champs_old.loc[champs_old['name'] == 'KhaZix', 'name_clean'] = 'khazix'
champs_old.loc[champs_old['name'] == 'Leblanc', 'name_clean'] = 'leblanc'
champs_old.loc[champs_old['name'] == 'VelKoz', 'name_clean'] = 'velkoz'

# Join old IDs to new metadata
champ_meta = champs_old.merge(champs_new[['name_clean', 'Classes', 'Role', 'Difficulty']], on='name_clean', how='left')

# Fill missing metadata with "Unknown" for old champs not in 2024 list (if any remain)
champ_meta['Classes'].fillna('Unknown', inplace=True)
champ_meta['Role'].fillna('Unknown', inplace=True)
champ_meta['Difficulty'].fillna('Unknown', inplace=True)

print("Champion metadata shape:", champ_meta.shape)


Champion metadata shape: (138, 6)


C:\Users\asus\AppData\Local\Temp\ipykernel_13752\984078587.py:20: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  champ_meta['Classes'].fillna('Unknown', inplace=True)
C:\Users\asus\AppData\Local\Temp\ipykernel_13752\984078587.py:21: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignm

## 3. Merge Datasets

In [4]:
# Concatenate stats1 and stats2 since they are row-wise splits of the same columns
stats = pd.concat([stats1, stats2], ignore_index=True)

# Merge participants with their stats
df = pd.merge(participants, stats, on='id', how='left')

# Drop duration from matches if we only need specific columns, but duration is crucial
# Merge with matches to get duration and game start time
df = pd.merge(df, matches[['id', 'duration', 'creation']], left_on='matchid', right_on='id', suffixes=('', '_match'))
df.rename(columns={'id_match': 'matchid_check'}, inplace=True)
df.drop('matchid_check', axis=1, inplace=True)

# Add team mapping (100 and 200 based on participant player 1-5 = 100, 6-10 = 200)
df['team_id'] = np.where(df['player'] <= 5, 100, 200)

# Merge with teamstats
teamstats_subset = teamstats[['matchid', 'teamid', 'firstblood', 'firsttower', 'firstinhib', 'firstbaron', 'firstdragon', 'firstharry', 'towerkills', 'inhibkills', 'baronkills', 'dragonkills', 'harrykills']]
df = pd.merge(df, teamstats_subset, left_on=['matchid', 'team_id'], right_on=['matchid', 'teamid'], how='left', suffixes=('', '_team'))
df.drop('teamid', axis=1, inplace=True)

# Merge with champion metadata
df = pd.merge(df, champ_meta[['id', 'name', 'Classes', 'Role', 'Difficulty']], left_on='championid', right_on='id', suffixes=('', '_champ'))
df.rename(columns={'name': 'champion_name', 'Classes': 'champion_classes', 'Role': 'champion_role', 'Difficulty': 'champion_difficulty'}, inplace=True)
if 'id_champ' in df.columns:
    df.drop('id_champ', axis=1, inplace=True)

print(f"Merged dataframe shape: {df.shape}")


Merged dataframe shape: (1834520, 81)


## 4. Final Cleanup & Outliers

In [5]:
# 1. Standardize column names (lowercase) FIRST
df.columns = [col.lower() for col in df.columns]

# 2. Filter out remakes (duration < 300 seconds)
df = df[df['duration'] >= 300].copy()

# 3. Handle outliers
# Cap extreme values
df['kills'] = df['kills'].clip(upper=df['kills'].quantile(0.999))
df['deaths'] = df['deaths'].clip(upper=df['deaths'].quantile(0.999))
df['assists'] = df['assists'].clip(upper=df['assists'].quantile(0.999))

# Check missing values
missing = df.isnull().sum()
print("Top missing values:\n", missing[missing > 0])

# Fill na for firstblood, firsttower, etc with 0 since it implies it didn't happen
cols_to_fill_0 = ['firstblood', 'firsttower', 'firstinhib', 'firstbaron', 'firstdragon', 'firstharry']
df[cols_to_fill_0] = df[cols_to_fill_0].fillna(0)

print(f"Final clean matches shape: {df.shape}")


Top missing values:
 win                          3
item1                        3
item2                        3
item3                        3
item4                        3
item5                        3
item6                        3
trinket                      3
kills                        3
deaths                       3
assists                      3
largestkillingspree          3
largestmultikill             3
killingsprees                3
longesttimespentliving       3
doublekills                  3
triplekills                  3
quadrakills                  3
pentakills                   3
legendarykills               3
totdmgdealt                  3
magicdmgdealt                3
physicaldmgdealt             3
truedmgdealt                 3
largestcrit                  3
totdmgtochamp                3
magicdmgtochamp              3
physdmgtochamp               3
truedmgtochamp               3
totheal                      3
totunitshealed               3
dmgselfmit        

## 5. Save Processed Data

In [6]:
# Save to processed directory
# We only save columns that are useful for EDA and modeling to save space
cols_to_keep = [
    'id', 'matchid', 'player', 'champion_name', 'champion_classes', 'champion_role', 'champion_difficulty',
    'ss1', 'ss2', 'role', 'position', 'win', 'kills', 'deaths', 'assists',
    'totdmgtochamp', 'totdmgdealt', 'totdmgtaken', 'goldearned', 'goldspent', 
    'totminionskilled', 'neutralminionskilled', 'visionscore', 'wardsplaced', 'wardskilled', 'duration',
    'team_id', 'firstblood', 'firsttower', 'firstbaron', 'firstdragon', 'towerkills', 'dragonkills', 'baronkills'
]

df_final = df[cols_to_keep]
df_final.to_csv(f'{OUTPUT_DIR}/clean_matches.csv', index=False)
print("Saved clean_matches.csv!")


Saved clean_matches.csv!
